# Populate Workspace Data

The suite of notebooks in this directory demonstrate how to perform a variety of Geostatistical modeling operations using Seequent Evo's compute task functionality. Prior to executing any compute tasks it is necessary to populate the Evo workspace you'll be using with data. 

## Requirements

You must have a Seequent account with the Evo entitlement. You'll need:
- The client ID of your Evo application
- The callback/redirect URL of your Evo application

To obtain these credentials, refer to the [Apps and tokens guide](https://developer.seequent.com/docs/guides/getting-started/apps-and-tokens).

---

First, import needed python packages including the [evo-python-sdk](https://github.com/SeequentEvo/evo-python-sdk) which facilities authentication, communication and data exchange from local computer to Seequent Evo.

In [ ]:
import pandas as pd

import evo.blockmodels.typed as evo_bm
import evo.objects.typed as evo_objs
from evo.notebooks import ServiceManagerWidget
from evo.widgets import get_viewer_url_for_objects

# Load the widgets extension for rich HTML display of Evo objects
%load_ext evo.widgets

In the next cell, authentication is performed using the `ServiceManagerWidget` which returns a python object that contains everything needed to connect to Evo services.

In [ ]:
evo_connection = await ServiceManagerWidget.with_auth_code(client_id="<CLIENT-ID>").login()

## First Upload the Drillholes

For this suite of examples, assume the data has already been composited and that the xyz and locations reprsent the midpoint of each composite itnerval. First, load the data from csv into a [pandas dataframe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) and inspect the first few rows.


In [ ]:
df = pd.read_csv("sample-data/WP_assay.csv")
df.head()

A quick statistical summary can be helpful, especially with unfamiliar data.

In [ ]:
df.describe()

## Upload Composites to Seequent Evo

In this case a [poinset object](https://developer.seequent.com/docs/data-structures/geoscience-objects/schemas/objects/pointset) is sensible because we've got spaitally located sample data without any from/to information. If povided from/to information, consider a [downhole intervals object](https://developer.seequent.com/docs/data-structures/geoscience-objects/schemas/objects/downhole-intervals).

To create the pointset object, first create the `PointSetData`, then upload to create a `PointSet` on Seequent Evo. Note that once created, the pointset can accessed on the Seequent Evo portal via the `Portal` and `Viewer` links.

In [ ]:
# create the data object locally
pointset_data = evo_objs.PointSetData(
    name="WP comps",
    description="Composited Cu and Au assays with Density",
    locations=df,
    coordinate_reference_system="EPSG:32650",
)

# upload and create the pointset within Seequent Evo workspace
pointset = await evo_objs.PointSet.create(evo_connection, pointset_data)
pointset

## Upload a Variogram Model

Variogram's can be created in Leapfrog Edge and published to Evo or can be created using known parameters. 

In this case the variogram model is constructued from two spherical structures that share the same rotation (`is_rotation_fixed=True`). This has been created in the original units (not standaridzed or normal score transformed) so establish that both the `sill` and the `data_variance` are equal.

To simplify the creation of multiple spherical strctures a small function is created that accepts 5 parameters including the contribution to the sill, the rotation, and the three ranges of that structure.

In [ ]:
import evo.objects.typed as evo_objs

In [ ]:
rotation = evo_objs.Rotation(
    dip=70.0,
    dip_azimuth=15.0,
    pitch=0.0,
)


def spherical_structure(contribution, rotation, major, semi_major, minor):
    return evo_objs.SphericalStructure(
        contribution=contribution,
        anisotropy=evo_objs.Ellipsoid(
            ranges=evo_objs.EllipsoidRanges(
                major=major,
                semi_major=semi_major,
                minor=minor,
            ),
            rotation=rotation,
        ),
    )


structures = [
    spherical_structure(0.25, rotation, 80.0, 60.0, 40.0),
    spherical_structure(0.51, rotation, 250.0, 180.0, 100.0),
]

Next, compile the structures along with other pertinant data and meta data into the `VariogramData` object for upload to the Seequent Evo workspace.

In [ ]:
variogram_data = evo_objs.VariogramData(
    name="Cu_pct Variogram Model",
    description="Variogram model for Cu_pct attribute in original units",
    sill=0.84,
    nugget=0.08,
    is_rotation_fixed=True,
    modelling_space="data",
    data_variance=0.84,
    attribute="Cu_pct",
    structures=structures,
)

# Now, create the variogram object in the Evo workspace
variogram = await evo_objs.Variogram.create(evo_connection, variogram_data)
variogram

## Create a Block Model

The last object needed is the blockmodel that will be the target of geostatistical modeling and contain the results. Similar to the above examples, first create teh `RegularBlockModelData` object, then upload to Evo to generate the `BlockModel` object

In [ ]:
blockmodel_data = evo_objs.RegularBlockModelData(
    name="WP Regular BlockModel",
    description="Regular block model for the WP deposit",
    origin=evo_objs.Point3(x=444750, y=492850, z=2350),
    n_blocks=evo_objs.Size3i(nx=50, ny=75, nz=45),
    block_size=evo_objs.Size3d(dx=20.0, dy=20.0, dz=20.0),
    size_unit_id=evo_bm.Units.METRES,
)

block_model = await evo_objs.BlockModel.create_regular(evo_connection, blockmodel_data)
block_model

Finally, visualize and inspect the objects using the Evo viewer.

In [ ]:
get_viewer_url_for_objects(evo_connection, [pointset, variogram, block_model])

Now that the Evo workspace has been populated with the necessary input composites and variogram along with a blockmodel to store results, continue on geostatistical modeling examples within this directory.